In [1]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import r2_score
from prophet import Prophet
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

In [2]:
veri = pd.read_csv("C:/Users/kerem/Desktop/Akıllı Trafik/Modeller/proje.csv")
veri = veri.copy()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda  x: '%.3f' % x)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

SARIMAX

In [3]:
# çevreyi etkileyen faktörler
def prepare_exog_full(veri):
    exog_cols = [
        "period_code",
        "haftanin_gunu",
        "ay",
        "rain_encoded",
        "NUMBER_OF_VEHICLES",
        "MINIMUM_SPEED",
        "MAXIMUM_SPEED",
        "AVERAGE_SPEED"
    ]
    return veri[exog_cols]


def sarimax_full_forecast_per_geohash(veri, test_size=720):

    results = []

    for geo, sub in veri.groupby("GEOHASH_SHORT"):

        sub = sub.sort_values(["DATE", "period_code"])
        sub = sub.set_index("DATE")

        if len(sub) <= test_size + 30:
            continue

        y = sub["DENSITY"]
        X = prepare_exog_full(sub)

        y_train = y.iloc[:-test_size]
        y_test  = y.iloc[-test_size:]

        X_train = X.iloc[:-test_size]
        X_test  = X.iloc[-test_size:]

        try:
            model = SARIMAX(
                endog=y_train,
                exog=X_train,
                order=(2,1,2),
                seasonal_order=(1,1,1,5),  # 5 period/day
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            res = model.fit(disp=False, maxiter=300)

            preds = res.forecast(steps=test_size, exog=X_test)

            mae = mean_absolute_error(y_test, preds)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            r2 = r2_score(y_test, preds)

            results.append({
                "GEOHASH": geo,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })

        except Exception as e:
            print(f"{geo} hata: {e}")

    return pd.DataFrame(results)

PROPHET

In [4]:
def prophet_full_forecast_per_geohash(veri, test_size=720):

    results = []

    for geo, sub in veri.groupby("GEOHASH_SHORT"):

        sub = sub.sort_values(["DATE", "period_code"])
        sub = sub.set_index("DATE")

        if len(sub) <= test_size + 30:
            continue

        y = sub["DENSITY"]
        X = prepare_exog_full(sub)

        full = pd.concat([y, X], axis=1).reset_index()
        full = full.rename(columns={"DATE": "ds", "DENSITY": "y"})

        train = full.iloc[:-test_size]
        test  = full.iloc[-test_size:]

        m = Prophet(
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=True,
            changepoint_prior_scale=0.5,
            seasonality_mode="multiplicative"
        )

        for col in [
            "period_code", "haftanin_gunu", "ay",
            "rain_encoded", "NUMBER_OF_VEHICLES",
            "MINIMUM_SPEED", "MAXIMUM_SPEED", "AVERAGE_SPEED"
        ]:
            m.add_regressor(col)

        m.fit(train)

        future = test[[
            "ds", "period_code", "haftanin_gunu", "ay",
            "rain_encoded", "NUMBER_OF_VEHICLES",
            "MINIMUM_SPEED", "MAXIMUM_SPEED", "AVERAGE_SPEED"
        ]]

        forecast = m.predict(future)
        preds = forecast["yhat"].values

        mae = mean_absolute_error(test["y"], preds)
        rmse = np.sqrt(mean_squared_error(test["y"], preds))
        r2 = r2_score(test["y"], preds)

        results.append({
            "GEOHASH": geo,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(results)

In [5]:
# ----------------------------
# 1) VERİYİ HAZIRLA
# ----------------------------
veri["DATE"] = pd.to_datetime(veri["DATE"], errors="coerce")

# yağış durumu encode
veri["rain_encoded"] = veri["rainfall_condition"].astype("category").cat.codes

# time period encode
veri["period_code"] = veri["TIME_PERIOD_NUM"]

# eksikleri at
veri = veri.dropna(subset=["DENSITY"])

# ----------------------------
# 2) SARIMAX TAHMİNLERİ
# ----------------------------
sarimax_results = sarimax_full_forecast_per_geohash(veri, test_size=168)
print("SARIMAX Sonuçları:")
print(sarimax_results.head())

# ----------------------------
# 3) PROPHET TAHMİNLERİ
# ----------------------------
prophet_results = prophet_full_forecast_per_geohash(veri, test_size=168)
print("\nProphet Sonuçları:")
print(prophet_results.head())

SARIMAX Sonuçları:
  GEOHASH   MAE  RMSE    R2
0   sx7ch 0.586 0.690 0.986
1   sx7ck 0.257 0.338 0.994
2   sx7cm 0.279 0.389 0.995
3   sx7cq 0.034 0.048 0.985
4   sx7ct 0.391 0.442 0.978


12:52:53 - cmdstanpy - INFO - Chain [1] start processing
12:52:54 - cmdstanpy - INFO - Chain [1] done processing
12:52:54 - cmdstanpy - INFO - Chain [1] start processing
12:52:55 - cmdstanpy - INFO - Chain [1] done processing
12:52:55 - cmdstanpy - INFO - Chain [1] start processing
12:52:56 - cmdstanpy - INFO - Chain [1] done processing
12:52:56 - cmdstanpy - INFO - Chain [1] start processing
12:52:57 - cmdstanpy - INFO - Chain [1] done processing
12:52:57 - cmdstanpy - INFO - Chain [1] start processing
12:52:58 - cmdstanpy - INFO - Chain [1] done processing
12:52:59 - cmdstanpy - INFO - Chain [1] start processing
12:52:59 - cmdstanpy - INFO - Chain [1] done processing
12:53:00 - cmdstanpy - INFO - Chain [1] start processing
12:53:01 - cmdstanpy - INFO - Chain [1] done processing
12:53:01 - cmdstanpy - INFO - Chain [1] start processing
12:53:01 - cmdstanpy - INFO - Chain [1] done processing
12:53:02 - cmdstanpy - INFO - Chain [1] start processing
12:53:02 - cmdstanpy - INFO - Chain [1]


Prophet Sonuçları:
  GEOHASH   MAE  RMSE     R2
0   sx7ch 3.267 3.936  0.558
1   sx7ck 1.065 1.296  0.915
2   sx7cm 3.199 3.700  0.572
3   sx7cq 0.389 0.453 -0.384
4   sx7ct 1.274 1.531  0.742
